In [1]:
pip install earthengine-api geemap --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 63.5 MB/s eta 0:00:0000:0100:01


# Using the OSM GEE account
## Cloud Project: osmgee
## Quota tier: Partner Tier (Until October)

In [ ]:
import ee
import geemap
ee.Authenticate()
# Initialize with your project
ee.Initialize(project='osmgee')

## The wetland and contributing watershed, provided by Dr. Zhibang Lv, have been uploaded to GEE assests.

In [ ]:
# ============================================
# STEP 1: Load assets and define study parameters
# ============================================

# Load the two feature collections — kept separate for later zonal extraction
wetlands = ee.FeatureCollection('projects/osmgee/assets/OSM_Wetland')
watersheds = ee.FeatureCollection('projects/osmgee/assets/OSM_Watersheds')

# Quick sanity checks
print('Wetland feature count:', wetlands.size().getInfo())
print('Watershed feature count:', watersheds.size().getInfo())
print('First wetland feature:', wetlands.first().getInfo())
print('First watershed feature:', watersheds.first().getInfo())

# ------------------------------------------------------------------
# Two distinct study areas, kept separate for later NDVI extraction:
#   - wetland_geom    : union of all 121 wetland polygons
#   - watershed_geom  : union of all 121 watershed polygons
# ------------------------------------------------------------------
wetland_geom = wetlands.geometry()
watershed_geom = watersheds.geometry()

# Combined bounding geometry — used ONLY to filter the Landsat collection
study_area = watershed_geom.union(wetland_geom).dissolve()

print('Wetland geometry type:', wetland_geom.type().getInfo())
print('Watershed geometry type:', watershed_geom.type().getInfo())

Wetland feature count: 121
Watershed feature count: 121
First wetland feature: {'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-111.70390746799144, 57.22722193224847], [-111.70391185654573, 57.22717299079201], [-111.70389666527063, 57.22715918120957], [-111.70384392025571, 57.227104209493945], [-111.70378135243628, 57.22706685491684], [-111.70374428383869, 57.22702126090118], [-111.70371543581047, 57.226975806688564], [-111.70368415209386, 57.22695712938566], [-111.70361273164801, 57.22692852811409], [-111.70358953405798, 57.22691001494814], [-111.7035381104251, 57.226841771953424], [-111.7035068269658, 57.22682309461844], [-111.70346662209924, 57.226813092483575], [-111.70342091502113, 57.2267716880284], [-111.70335834845145, 57.22673433330282], [-111.70333486895083, 57.226720305742965], [-111.70329466430755, 57.226710303547996], [-111.70326344947281, 57.226691704283525], [-111.70324073726775, 57.22666875921929], [-111.70315517682813, 57.22661294497201], [-111.70

## Focusing only on the summer composites for clear observations

In [ ]:
# ============================================
# Study period and growing season window
# ============================================
start_year = 1985
end_year = 2025

start_month = 6   # June
end_month = 8     # August

In [46]:
# ============================================
# STEP 2: Build harmonized Landsat collection (1985-2025)
# ============================================

"""Cloud/shadow/snow mask for Landsat 4/5/7 Collection 2 SR."""
def mask_l457(image):
    qa = image.select('QA_PIXEL')
    # Bits: 1=Dilated Cloud, 3=Cloud, 4=Cloud Shadow, 5=Snow
    cloud_shadow_bit = 1 << 4
    clouds_bit = 1 << 3
    snow_bit = 1 << 5
    mask = (qa.bitwiseAnd(cloud_shadow_bit).eq(0)
            .And(qa.bitwiseAnd(clouds_bit).eq(0))
            .And(qa.bitwiseAnd(snow_bit).eq(0)))
    return image.updateMask(mask)


def mask_l89(image):
    """Cloud/shadow/snow mask for Landsat 8/9 Collection 2 SR."""
    qa = image.select('QA_PIXEL')
    cloud_shadow_bit = 1 << 4
    clouds_bit = 1 << 3
    snow_bit = 1 << 5
    mask = (qa.bitwiseAnd(cloud_shadow_bit).eq(0)
            .And(qa.bitwiseAnd(clouds_bit).eq(0))
            .And(qa.bitwiseAnd(snow_bit).eq(0)))
    return image.updateMask(mask)


"""Harmonize Landsat 4/5/7 band names to common names."""
def rename_l457(image):
    """Harmonize Landsat 4/5/7 band names to common names."""
    return image.select(
        ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'],
        ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']
    ).copyProperties(image, ['system:time_start', 'SPACECRAFT_ID'])


def rename_l89(image):
    """Harmonize Landsat 8/9 band names to common names."""
    return image.select(
        ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
        ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']
    ).copyProperties(image, ['system:time_start', 'SPACECRAFT_ID'])


def apply_scale_factors(image):
    """Apply Collection 2 SR scale factors to get true reflectance."""
    optical_bands = image.select(
        ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']
    ).multiply(0.0000275).add(-0.2)
    return image.addBands(optical_bands, None, True)


# Load each sensor's Collection 2 Level-2 SR collection, filtered to study area + date range
l5 = (ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
      .filterBounds(study_area)
      .filterDate('1985-01-01', '2013-01-01')
      .map(mask_l457)
      .map(rename_l457))

l7 = (ee.ImageCollection('LANDSAT/LE07/C02/T1_L2')
      .filterBounds(study_area)
      .filterDate('1999-01-01', '2022-01-01')  # SLC-off after 2003, but keep for gap-fill overlap
      .map(mask_l457)
      .map(rename_l457))

l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
      .filterBounds(study_area)
      .filterDate('2013-01-01', '2025-12-31')
      .map(mask_l89)
      .map(rename_l89))

l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
      .filterBounds(study_area)
      .filterDate('2021-10-01', '2025-12-31')
      .map(mask_l89)
      .map(rename_l89))

# Merge all sensors into one collection, apply scale factors, keep SPACECRAFT_ID
landsat_all = (l5.merge(l7).merge(l8).merge(l9)
               .map(apply_scale_factors))

print('Total merged Landsat scenes:', landsat_all.size().getInfo())

# Sanity check — inspect one image's bands and a property
first_img = landsat_all.first()
print('Band names:', first_img.bandNames().getInfo())
print('SPACECRAFT_ID:', first_img.get('SPACECRAFT_ID').getInfo())

Total merged Landsat scenes: 6724
Band names: ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']
SPACECRAFT_ID: LANDSAT_5


## 50% Cloud cover has been used.

In [ ]:
# ============================================
# Filter to growing season + sort by cloud cover
# ============================================

def add_cloud_cover_property(image):
    """Ensure CLOUD_COVER is accessible as a numeric property for sorting/filtering."""
    return image.set('CLOUD_COVER', ee.Number(image.get('CLOUD_COVER')))


# Filter the merged collection to the growing season window (across all years)
landsat_growing_season = (landsat_all
    .filter(ee.Filter.calendarRange(start_month, end_month, 'month'))
    .filter(ee.Filter.calendarRange(start_year, end_year, 'year'))
    .map(add_cloud_cover_property))

# Optional: drop scenes above a cloud cover threshold before compositing
# (adjust threshold based on how much data you have available in this region —
#  AOSR has persistent cloud cover per your Project Report Section 2.5, so a
#  very strict threshold may over-thin the collection)
MAX_CLOUD_COVER = 50  # percent

landsat_filtered = landsat_growing_season.filter(
    ee.Filter.lte('CLOUD_COVER', MAX_CLOUD_COVER)
)

# Sort by cloud cover (ascending — least cloudy first)
# Note: sorting an ImageCollection matters most when you later use
# qualityMosaic(), mosaic(), or .first() — median() composites are
# order-independent, so sorting mainly helps downstream flexibility
landsat_sorted = landsat_filtered.sort('CLOUD_COVER')

print('Scenes before cloud filter:', landsat_growing_season.size().getInfo())
print('Scenes after cloud filter (<= {}%):'.format(MAX_CLOUD_COVER),
      landsat_filtered.size().getInfo())

# Sanity check — confirm sort worked, look at cloud cover of first few images
cloud_values = landsat_sorted.limit(5).aggregate_array('CLOUD_COVER').getInfo()
print('Cloud cover of 5 least-cloudy scenes:', cloud_values)

Scenes before cloud filter: 2467
Scenes after cloud filter (<= 50%): 1402
Cloud cover of 5 least-cloudy scenes: [0, 0, 0, 0, 0]


In [ ]:
def clip_to_study_area(image):
    """Clip image to study area to reduce computation to only relevant pixels."""
    return image.clip(study_area)


# Apply clipping when building landsat_all, or as an additional .map() step
landsat_all_clipped = landsat_all.map(clip_to_study_area)


# Cross Sensor harmonization is not needed 

## (https://gis.stackexchange.com/questions/440988/landsat-collection-2-surface-reflectance-harmonization
## https://developers.google.com/earth-engine/faq#is_cross-sensor_landsat_surface_reflectance_harmonization_needed)

In [8]:
# ============================================
# Filter clipped collection to growing season, compute NDVI
# ============================================

def add_ndvi(image):
    """Compute NDVI and add as a band."""
    ndvi = image.normalizedDifference(['NIR', 'Red']).rename('NDVI')
    return image.addBands(ndvi)


landsat_ndvi = (landsat_all_clipped
    .filter(ee.Filter.calendarRange(start_month, end_month, 'month'))
    .map(add_ndvi))

print('Scenes in June-Aug window (all years):', landsat_ndvi.size().getInfo())

Scenes in June-Aug window (all years): 2467


In [9]:
# ============================================
# STEP 3: Annual NDVI-only medoid composites (June-August, 1985-2025)
# ============================================

def medoid_composite_ndvi(collection):
    """
    Build a medoid composite using ONLY the NDVI band.
    For each pixel, selects the actual observed NDVI value closest
    to the median NDVI across all available scenes that year.
    """
    median_ndvi = collection.select('NDVI').median()

    def add_distance(image):
        distance = image.select('NDVI').subtract(median_ndvi).abs().rename('distance')
        return image.addBands(distance)

    with_distance = collection.map(add_distance)

    def negate_distance(image):
        return image.addBands(image.select('distance').multiply(-1).rename('neg_distance'))

    with_neg_distance = with_distance.map(negate_distance)

    medoid = with_neg_distance.qualityMosaic('neg_distance')

    return medoid.select('NDVI')


def make_annual_composite(year):
    """Build one annual NDVI-only medoid composite for a given year."""
    year = ee.Number(year)
    start = ee.Date.fromYMD(year, start_month, 1)
    end = ee.Date.fromYMD(year, end_month, 31)

    yearly_scenes = landsat_ndvi.filterDate(start, end.advance(1, 'day'))

    composite = (medoid_composite_ndvi(yearly_scenes)
        .set('year', year)
        .set('system:time_start', start.millis())
        .set('scene_count', yearly_scenes.size()))

    return composite

In [ ]:
# Test on a single year first
test_composite = make_annual_composite(1998)

print('Test composite bands:', test_composite.bandNames().getInfo())
print('Test composite scene count:', test_composite.get('scene_count').getInfo())

test_thumb_url = test_composite.select('NDVI').getThumbURL({
    'min': -0.2,
    'max': 0.9,
    'palette': ['brown', 'yellow', 'green', 'darkgreen'],
    'dimensions': 512,
    'region': study_area,
})

from IPython.display import Image as IPImage
IPImage(url=test_thumb_url)

Test composite bands: ['NDVI']
Test composite scene count: 64


In [78]:
# ============================================
# Export test composite (1998 NDVI medoid) to GEE Asset
# ============================================

export_task_asset = ee.batch.Export.image.toAsset(
    image=test_composite,
    description='NDVI_medoid_1998_test',
    assetId='projects/annual-ccdc-new/assets/NDVI_medoid_1998_test',
    region=study_area,
    scale=30,
    maxPixels=1e9
)

export_task_asset.start()
print('Export to Asset started. Task ID:', export_task_asset.id)

Export to Asset started. Task ID: A2BTVCUIHSYZP7GYBLRYTQY2


In [17]:
import time

task = export_task_asset  # or export_task_drive
while task.active():
    print('Task status:', task.status()['state'])
    time.sleep(30)

print('Final status:', task.status())

Task status: READY
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING


Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: RUNNING
Task status: 